# LinkedIn Career Agent

**By: Dario Melconian**

A conversational AI agent that represents me on my website — answering questions about my career, projects, and background as if I were there in person.

Built with Claude Sonnet as the chat model and GPT-4o-mini as an independent evaluator. If a response doesn't pass quality control, it's automatically regenerated with the rejection feedback included.

### Agentic Patterns Used:

| Pattern | Description |
|---|---|
| **LLM-as-Judge** | GPT-4o-mini evaluates every Claude response before it reaches the user |
| **Iterative Refinement** | Rejected responses are retried with the evaluator's feedback injected into the prompt |
| **RAG (lightweight)** | LinkedIn PDF, STAR stories, project docs, and GitHub repos all loaded as context |

### Model Setup:

| Role | Model | Provider |
|---|---|---|
| Chat Agent | `claude-sonnet-4-5` | Anthropic |
| Evaluator | `gpt-4o-mini` | OpenAI |

In [242]:
import os
import requests
import base64
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from pypdf import PdfReader
import gradio as gr

In [243]:
load_dotenv(override=True)
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
claude = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
MODEL = "claude-sonnet-4-5"

## Step 1 — Load Personal Context

Read in all the personal documents that will be used to ground the agent's responses: LinkedIn profile, personal summary, STAR project stories, additional project details, and the NLP course syllabus from UofT.

In [244]:
reader = PdfReader("me/linkedin.pdf") # read in my profile, the pages
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [245]:
print(linkedin)

   
Contact
d.melconian@outlook.com
www.linkedin.com/in/
dariomelconian (LinkedIn)
medium.com/@dario61melconian
(Blog)
Top Skills
Transfer Learning
Large Language Models (LLM)
Natural Language Processing (NLP)
Languages
Italian
Spanish (Limited Working)
Certifications
Develop Generative AI Applications
2nd Place - DataQuest AI
Competition
Build RAG Applications
Applied Natural Language
Processing
Process Mining Course
Dario Melconian
Data Scientist at BMO Private Wealth
Toronto, Ontario, Canada
Summary
Data Scientist and AI practitioner with 4+ years of experience
building production-grade machine learning and NLP systems
at the intersection of technical depth and business impact. At
BMO, operating as both hands-on data scientist and internal
data science consultant, translating complex analytical problems
into scalable solutions for executive stakeholders across Wealth
Management.Work spans the full model lifecycle: from feature
engineering and ML pipeline development to deployment an

In [246]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [247]:
# take some PDF/docx files as context for the agent regarding details about my projects
from docx import Document

def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() for page in reader.pages if page.extract_text())

def read_docx(path):
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

stars = read_pdf("me/INTRO_STARS.pdf")
stripe = read_docx("me/Stripe_R1.docx")
nlp_syllabus = read_pdf("me/3666-018_OL_Applied Natural Language Processing.pdf")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 77 0 (offset 0)
Ignoring wrong pointing object 79 0 (offset 0)
Ignoring wrong

### Fetch GitHub Repositories

Pull all public repos and their READMEs from GitHub to give the agent full context on current projects. Requires a `GITHUB_TOKEN` in `.env` to avoid the 60 req/hr unauthenticated rate limit.

In [248]:
def fetch_github_repos(username):
    token = os.getenv("GITHUB_TOKEN")
    headers = {"Authorization": f"token {token}"} if token else {}

    repos_url = f"https://api.github.com/users/{username}/repos?per_page=100&sort=updated"
    response = requests.get(repos_url, headers=headers)
    repos = response.json()

    if not isinstance(repos, list):
        print(f"GitHub API error: {repos.get('message', repos)}")
        return ""

    github_content = ""
    for repo in repos:
        if repo.get("fork"):
            continue
        name = repo["name"]
        description = repo.get("description") or ""
        url = repo["html_url"]
        github_content += f"### {name}\n**URL:** {url}\n**Description:** {description}\n"

        readme_url = f"https://api.github.com/repos/{username}/{name}/readme"
        readme_resp = requests.get(readme_url, headers=headers)
        if readme_resp.status_code == 200:
            readme_text = base64.b64decode(readme_resp.json()["content"]).decode("utf-8")
            github_content += f"**README:**\n{readme_text}\n"
        github_content += "\n---\n\n"

    return github_content

github = fetch_github_repos("dariomelconian")
if github:
    print(f"Fetched GitHub repos ({len(github)} chars)")

Fetched GitHub repos (25856 chars)


## Step 2 — Build the System Prompt

Assemble all loaded context into a single system prompt that tells Claude who it is, how to behave, and everything it needs to answer questions accurately.

In [249]:
name = "Dario Melconian"

Basic Prompt that initially can't handle unknown answers.

In [250]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background, LinkedIn profile, GitHub repositories, STAR project stories, \
and relevant coursework which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n"
system_prompt += f"## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"## GitHub Repositories:\n{github}\n\n"
system_prompt += f"## STAR Project Stories:\n{stars}\n\n"
system_prompt += f"## Additional Project Details:\n{stripe}\n\n"
system_prompt += f"## NLP Coursework (UofT):\n{nlp_syllabus}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [251]:
system_prompt

'You are acting as Dario Melconian. You are answering questions on Dario Melconian\'s website, particularly questions related to Dario Melconian\'s career, background, skills and experience. Your responsibility is to represent Dario Melconian for interactions on the website as faithfully as possible. You are given a summary of Dario Melconian\'s background, LinkedIn profile, GitHub repositories, STAR project stories, and relevant coursework which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Dario Melconian. I\'m a data scientist and AI engineer. I am from Toronto, Canada.\nIn my past time, I love sports, food, and travel (preferably somewhere warm).\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\nd.melconian@outlook.com\nwww.linkedin.com/in/\ndariomelconian (LinkedIn)\nmedium.com/@dario61melconian\n(Blog)\nTop Skills\nTrans

## Step 3 — Evaluator Setup (GPT-4o-mini)

The evaluator runs independently from the chat agent. It receives the same full context and grades each response before it reaches the user. Using a different model here intentionally avoids self-evaluation bias.

### Test — Single Response + Evaluation

Run a one-off question through Claude and then pass the reply to the evaluator to verify the full pipeline works end to end.

In [252]:
# Create Pydantic model for evaluation of the agent's responses
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [253]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable or not. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is of acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their LinkedIn and GitHub websites. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary, LinkedIn profile, GitHub repositories, STAR project stories, and relevant coursework. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n"
evaluator_system_prompt += f"## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"## GitHub Repositories:\n{github}\n\n"
evaluator_system_prompt += f"## STAR Project Stories:\n{stars}\n\n"
evaluator_system_prompt += f"## Additional Project Details:\n{stripe}\n\n"
evaluator_system_prompt += f"## NLP Coursework (UofT):\n{nlp_syllabus}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [254]:
# take the latest message, the agent's reply, and the conversation history, 
# and create a prompt for the evaluator
def evaluator_user_prompt(reply, message, history):     
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [255]:
EVALUATOR_MODEL = "gpt-4o-mini"

In [256]:
# object to evaluate the agent's response using the evaluator model
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.beta.chat.completions.parse(model=EVALUATOR_MODEL, messages=messages, response_format=Evaluation) # Pydantic object to be populated (JSON)
    return response.choices[0].message.parsed

In [257]:
messages = [{"role": "user", "content": "Tell me about your experience at BMO."}]
response = claude.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=system_prompt,
    messages=messages
)
reply = response.content[0].text
print(reply)

Thanks for asking! My experience at BMO has been really rewarding and has allowed me to grow significantly as a data scientist.

I joined BMO Private Wealth in May 2022 as a Junior Data Scientist and was promoted to Data Scientist in April 2023, where I've been operating for about 3 years now. What makes my role unique is that I wear two hats - I'm both a **hands-on data scientist** building production-grade ML systems and an **internal data science consultant**, translating complex analytical problems into scalable solutions for executive stakeholders across Wealth Management.

I've worked on several high-impact projects:

**NLP Summarization Engine**: I built an automated system to process 200K+ unstructured call logs from client call centers. Using NLTK, TextRank, and custom preprocessing, I created actionable bullet-point summaries of the most recent 3 calls per client. This reduced manual review time by 80% (from 3-5 minutes to under 1 minute) and was rolled out to 500+ frontline 

In [258]:
# evaluation from the evaluator model, gpt-4o-mini, which will return a JSON 
# object that can be parsed into the Evaluation Pydantic model
evaluate(reply, "Tell me about your experience at BMO.", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is acceptable as it is professional, engaging, and provides a clear overview of the Agent's experience at BMO. It effectively highlights key projects and achievements, showcasing technical skills and their practical impacts in a way that would interest potential clients or employers. Additionally, the invitation for further questions encourages an engaging dialogue, which is beneficial in a client or employer context.")

In [259]:
def clean_messages(history):
    result = []
    for m in history:
        if isinstance(m, dict):
            result.append({"role": m["role"], "content": m["content"]})
        else:
            result.append({"role": m.role, "content": m.content})
    return result

In [260]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = clean_messages(history) + [{"role": "user", "content": message}]
    response = claude.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=updated_system_prompt,
        messages=messages
    )
    return response.content[0].text

## Step 4 — Chat Agent + Gradio UI

Wire everything together. The `chat` function calls Claude, passes the reply to the evaluator, and if it fails, calls `rerun` with the rejection feedback so Claude can try again. Gradio wraps it in a UI.

In [261]:
def chat(message, history):
    if "Quit" in message:
        system = system_prompt + "\n\nGreet the person a formal goodbye if they initiate it \
              make sure to thank them for visiting the website and encourage them to reach out if they please"
    else:
        system = system_prompt
    messages = clean_messages(history) + [{"role": "user", "content": message}]
    response = claude.messages.create(model=MODEL, max_tokens=1024, system=system, messages=messages)
    reply = response.content[0].text

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [262]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


---

## Summary of Agentic Patterns Used

| Pattern | Where Used |
|---|---|
| **LLM-as-Judge** | GPT-4o-mini evaluates every Claude response before returning it to the user |
| **Iterative Refinement** | Failed responses are retried with evaluator feedback injected into the system prompt |
| **RAG (lightweight)** | LinkedIn, GitHub, STAR stories, and coursework loaded as static context at startup |

-------

## Use Tools to Deploy Agent Professionally

- tool use = JSON & IF statements (put simply)
- use Pushover to send notifications to phone
- API setup for push noti to phone (part of career agent)

In [263]:
import json

In [265]:
# pushover credentials and endpoint
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# user check
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

# token check
if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [266]:
# push notification
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [267]:
# testing lol
push("HEY!")

Push: HEY!


In [268]:
# equip LLM if user wants to be in touch (email, name, notes)
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [269]:
# if unknown question, record it so I can answer it later and improve the agent thru time
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [277]:
# JSON that refers to the user detail function, calling it, why it can be used 
# sent to LLM so it can call the function when needed, describing the function for the LLM
# so it can decide if it wants to use/call this tool

# Anthropic tools format
record_user_details_json = [
    {
        "name": "record_user_details",
        "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
        "input_schema": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The email address of this user"},
                "name": {"type": "string", "description": "The user's name, if they provided it"},
                "notes": {"type": "string", "description": "Any additional information about the conversation that's worth recording to give context"}
            },
            "required": ["email"]
        }
    },
    {
        "name": "record_unknown_question",
        "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
        "input_schema": {
            "type": "object",
            
            "properties": {
                "question": {"type": "string", "description": "The question that couldn't be answered"}
            },
            "required": ["question"]
        }
    }
]

In [271]:
# explain to LLM when to use this tool, if the user asks q that agent doesn't know answer to
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [272]:
# compartmentalize the tools/functions that the LLM can call
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

- chunk of JSON describing the 2 functions provided with name, description, parameters
- LLMs loved JSON (help communication and interaction)
- option to reply, opting to say it wants to run one of the tools created
- need to run a function that handles situation when LLM wants to use tool

In [274]:
# globals lets you look up any function in scope
globals()["record_unknown_question"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


{'recorded': 'ok'}

In [278]:
# takes tool calls, runs them, and adds responses to results it returns
# this function can take a list of tool calls, and runs them. IF statement.
# This is a more elegant way that avoids the IF statement.
# Anthropic tool calls come as content blocks with type "tool_use"
# block.input is already a dict (no json.loads needed), block.id and block.name differ from OpenAI
def handle_tool_calls_claude(tool_use_blocks):
    results = []
    for block in tool_use_blocks:
        tool_name = block.name
        arguments = block.input
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result)})
    return results

In [279]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Dario Melconian"

Now prompt LLM how to deal with unknown answers, or getting in touch.

In [280]:
# technically, JSON has this info, but good practice to repeat to incr. probability
# of outputting tokens consistent with my objective.
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background in the form of their summary, LinkedIn profile, GitHub repositories, STAR project stories, and relevant coursework which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [282]:
def chat(message, history):
    clean_history = [{"role": m["role"], "content": m["content"]} for m in history if isinstance(m, dict)]
    messages = clean_history + [{"role": "user", "content": message}]
    done = False
    while not done:
        # call Claude API with tools enabled
        response = claude.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system_prompt,
            # now passing tools (JSON) describing what it can now do
            tools=record_user_details_json,
            messages=messages
        )
        if response.stop_reason == "tool_use":
            tool_use_blocks = [block for block in response.content if block.type == "tool_use"]
            results = handle_tool_calls_claude(tool_use_blocks)
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": results})
        else:
            done = True
    return next(block.text for block in response.content if hasattr(block, "text"))

In [ ]:
# relaunch now with tools implemented
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


### Deployment